# L5: Automate Event Planning

In this lesson, you will learn more about Tasks.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Crew, Task

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from com.example.agentic.utils import get_openai_api_key,get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'llama3.2:1b-instruct-q8_0'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [6]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipmen"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [7]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://docs.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [8]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

- By setting `async_execution=True`, it means the task can run in parallel with the tasks which come after it.

In [17]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [18]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="outputs/L010/marketing_report_{run_id}.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [19]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="outputs/L010/venue_details_{run_id}.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator,
    context=[logistics_task, marketing_task],
)

## Creating the Crew

**Note**: Since you set `async_execution=True` for `logistics_task` and `marketing_task` tasks, now the order for them does not matter in the `tasks` list.

In [20]:
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[logistics_manager, 
            marketing_communications_agent,venue_coordinator],
    
    tasks=[logistics_task, 
           marketing_task,venue_task],
    
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [21]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall",
    'run_id':run_id
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [ ]:
result = event_management_crew.kickoff(inputs=event_details)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 36aa0a6e-9b5d-4ee3-a810-78b8c2232bd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Coordinate catering and equipment for an event with 500 participants on 2024-09-15.                      │
│  ID: bc55f010-9413-4c68-b81a-9bde5196b848                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Promote the Tech Innovation Conference aiming to engage at least500 potential attendees.                 │
│  ID: 06d2782a-8546-4238-8c75-995b355a286c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│  Task: Promote the Tech Innovation Conference aiming to engage at least500 potential attendees.                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Logistics Manager                                                                                       │
│                                                                                                                 │
│  Task: Coordinate catering and equipment for an event with 500 participants on 2024-09-15.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 🔧 Tool Error (#11) ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: search_the_internet_with_serper                                                                          │
│  Iteration: 11                                                                                                  │
│  Attempt: 0                                                                                                     │
│  Error: Tool 'Search the internet with Serper' arguments validation failed: 1 validation error for              │
│  SerperDevToolSchema                                                                                            │
│  search_query                                                                                                   │
│    Field required [type=missing, input_value={'A': '', 'type': 'string'}, input_type=dict]                      │
│      For further information visit https://errors.pydantic.dev/2.11/v/missing                                   │
│  Expected arguments: {"search_query": {"description": "Mandatory search query you want to use to search the     │
│  internet", "title": "Search Query", "type": "string"}}                                                         │
│  Required: ["search_query"]                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🔧 Tool Error (#5) ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: read_website_content                                                                                     │
│  Iteration: 5                                                                                                   │
│  Attempt: 0                                                                                                     │
│  Error: Tool 'Read website content' arguments validation failed: 1 validation error for                         │
│  ScrapeWebsiteToolSchema                                                                                        │
│  website_url                                                                                                    │
│    Field required [type=missing, input_value={'Website': {}}, input_type=dict]                                  │
│      For further information visit https://errors.pydantic.dev/2.11/v/missing                                   │
│  Expected arguments: {"website_url": {"description": "Mandatory website url to read the file", "title":         │
│  "Website Url", "type": "string"}}                                                                              │
│  Required: ["website_url"]                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│  Args: {'website_url': 'https://eventinfo.com/event-logistics'}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_website_content executed with result: Error executing tool: HTTPSConnectionPool(host='eventinfo.com', port=443): Max retries exceeded with url: /event-logistics (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at ...

╭────────────────────────────────────────────── 🔧 Tool Error (#6) ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: read_website_content                                                                                     │
│  Iteration: 6                                                                                                   │
│  Attempt: 0                                                                                                     │
│  Error: HTTPSConnectionPool(host='eventinfo.com', port=443): Max retries exceeded with url: /event-logistics    │
│  (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7af85c24b9d0>, 'Connection to   │
│  eventinfo.com timed out. (connect timeout=15)'))                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the generated `venue_details.json` file.

In [ ]:
import json
from pprint import pprint

with open(f'outputs/L010/venue_details_{run_id}.json') as f:
   data = json.load(f)

pprint(data)

- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [ ]:
from IPython.display import Markdown
Markdown(f"outputs/L010/marketing_report_{run_id}.md")